In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/"

if os.path.exists(project_path):
    print("Path found:", project_path)
    print("\nContents:")
    for item in os.listdir(project_path):
        print("-", item)
else:
    print("Path not found:", project_path)

Mounted at /content/drive
Path found: /content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/

Contents:
- POLI3148_Assignment1_Instruction.pdf
- Data
- Code
- Report
- Prompt


# 1. Domestic Violence and Fatalities in Iran (2016-2026)

## Process Data

In [ ]:
import pandas as pd
import os
import glob

data_raw_path = os.path.join(project_path, "Data", "Data_Raw")
data_processed_path = os.path.join(project_path, "Data", "Data_Processed")

files = [
    "number_of_demonstration_events_by_country-year_as-of-03Ap.xlsx",
    "number_of_events_targeting_civilians_by_country-year_as-o.xlsx",
    "number_of_political_violence_events_by_country-year_as-of-03Apr2026.xlsx",
    "number_of_reported_civilian_fatalities_by_country-year_as.xlsx",
    "number_of_reported_fatalities_by_country-year_as-of-03Apr.xlsx"
]

all_iran_data = []

for file in files:
    file_path = os.path.join(data_raw_path, file)
    if os.path.exists(file_path):
        df = pd.read_excel(file_path)
        
        # Find the exact names of the required columns in this specific file
        country_col = [c for c in df.columns if c.lower() == 'country']
        year_col = [c for c in df.columns if c.lower() == 'year']
        val_col = [c for c in df.columns if c.upper() in ['EVENTS', 'FATALITIES']]
        
        if country_col and year_col and val_col: # Make sure we found all necessary parts
            # Filter for Iran
            df_iran = df[df[country_col[0]] == 'Iran'].copy()
            
            # Create a clean DataFrame strictly with the 4 columns requested
            clean_df = pd.DataFrame({
                'Country': df_iran[country_col[0]],
                'Year': df_iran[year_col[0]],
                'NUMBER': df_iran[val_col[0]],
                'Source_File': file
            })
            
            all_iran_data.append(clean_df)
        else:
            print(f"Missing required columns in: {file}")
            print(f"Found columns: {df.columns.tolist()}")
    else:
        print(f"File not found: {file_path}")

if all_iran_data:
    final_df = pd.concat(all_iran_data, ignore_index=True)
    out_path = os.path.join(data_processed_path, "Iran_violence.csv")
    final_df.to_csv(out_path, index=False)
    print(f"Data successfully filtered and consolidated to {out_path}")
    print(final_df.head(3)) # Show a quick preview to confirm the 4 columns
else:
    print("No data extracted.")

Data successfully filtered and consolidated to /content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/Data/Data_Processed/Iran_violence.csv
  Country  Year  NUMBER                                        Source_File
0    Iran  2016    1606  number_of_demonstration_events_by_country-year...
1    Iran  2017    2430  number_of_demonstration_events_by_country-year...
2    Iran  2018    3001  number_of_demonstration_events_by_country-year...


## Visualization

In [73]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import os

if os.path.exists(out_path):
    ir_df = pd.read_csv(out_path)
    
    # Check what columns exist to plot Yearly aggregation
    year_col = [c for c in ir_df.columns if 'year' in c.lower()]
    val_cols = [c for c in ir_df.columns if c.lower() not in ['country', 'iso', 'region', 'source_file', 'year']]
    
    if year_col and val_cols:
        y_col = year_col[0]
        numeric_cols = ir_df.select_dtypes(include='number').columns
        val_sum = [c for c in numeric_cols if c != y_col]
        
        yearly_data = ir_df.groupby([y_col, 'Source_File'])[val_sum].sum().reset_index()
        
        # --- Add Strategic Developments from Iran_disobedience.csv ---
        csv_file_path = os.path.join(project_path, "Data", "Data_Processed", "Iran_disobedience.csv")
        if os.path.exists(csv_file_path):
            disob_df = pd.read_csv(csv_file_path)
            if 'WEEK' in disob_df.columns and 'EVENT_TYPE' in disob_df.columns:
                disob_df['year_extracted'] = pd.to_datetime(disob_df['WEEK']).dt.year
                strat_df = disob_df[disob_df['EVENT_TYPE'].str.contains('Strategic developments', na=False, case=False)]
                strat_yearly = strat_df.groupby('year_extracted')['EVENTS'].sum().reset_index()
                strat_yearly['Source_File'] = 'strategic_developments'
                strat_yearly.rename(columns={'year_extracted': y_col, 'EVENTS': val_sum[0]}, inplace=True)
                yearly_data = pd.concat([yearly_data, strat_yearly], ignore_index=True)
        # -------------------------------------------------------------

        def get_clean_label(filename):
            name = filename.lower()
            if 'demonstration' in name: return 'Demonstration Events'
            elif 'civilians' in name and 'fatalities' not in name: return 'Events Targeting Civilians'
            elif 'political_violence' in name: return 'Political Violence Events'
            elif 'strategic_developments' in name: return 'Strategic Developments'
            elif 'civilian_fatalities' in name: return 'Reported Civilian Fatalities'
            elif 'reported_fatalities' in name: return 'Total Fatalities'
            return filename[:20]
        
        # Explicitly order the sources: demonstration, violence, strategic, targeting civilians, then fatalities
        ordered_keywords = [
            'demonstration',
            'political_violence',
            'strategic_developments',
            'targeting_civilians',
            'reported_fatalities',
            'civilian_fatalities'
        ]
        
        unique_sources = []
        for kw in ordered_keywords:
            for sf in yearly_data['Source_File'].unique():
                if kw in sf.lower() and sf not in unique_sources:
                    unique_sources.append(sf)
        # Catch any remaining ones just in case
        for sf in yearly_data['Source_File'].unique():
            if sf not in unique_sources:
                unique_sources.append(sf)
                
        # Calculate maximum Y value for the "events" datasets to share the same scale
        event_sfs = [sf for sf in unique_sources if any(k in sf.lower() for k in ['demonstration', 'political_violence', 'strategic', 'targeting_civilians'])]
        max_events_y = yearly_data[yearly_data['Source_File'].isin(event_sfs)][val_sum[0]].max() if event_sfs else 0
        
        # Calculate shared Y value bounds for the two "fatalities" datasets
        fatalities_sfs = [sf for sf in unique_sources if sf not in event_sfs]
        if fatalities_sfs:
            fat_subset = yearly_data[yearly_data['Source_File'].isin(fatalities_sfs)][val_sum[0]]
            min_fat_y = fat_subset.min()
            max_fat_y = fat_subset.max()
            fat_y_margin = (max_fat_y - min_fat_y) * 0.15 if max_fat_y > min_fat_y else max_fat_y * 0.15
            shared_fat_bottom = max(0, min_fat_y - fat_y_margin)
            shared_fat_top = max_fat_y + fat_y_margin
        else:
            shared_fat_bottom, shared_fat_top = 0, 100
        
        # ACLED-like muted regional colors extended to 6
        colors = ['#62778d', '#8a4053', '#8e44ad', '#719e9c', '#cc5c4b', '#488c5e']
        
        def hex_to_rgba(hex_color, alpha=0.15):
            hex_color = hex_color.lstrip('#')
            return f"rgba({int(hex_color[0:2], 16)}, {int(hex_color[2:4], 16)}, {int(hex_color[4:6], 16)}, {alpha})"

        # Determine grid size (e.g., 2 rows x 3 columns for 6 items)
        n_cols = 3
        n_rows = int(np.ceil(len(unique_sources) / n_cols))
        
        subplot_titles = [get_clean_label(sf) for sf in unique_sources]
        
        # Reduce horizontal spacing since inner Y-axes will be hidden
        fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=subplot_titles, vertical_spacing=0.2, horizontal_spacing=0.04)
        
        for idx, sf in enumerate(unique_sources):
            row = (idx // n_cols) + 1
            col = (idx % n_cols) + 1
            
            subset = yearly_data[yearly_data['Source_File'] == sf].sort_values(y_col)
            
            if not subset.empty and len(val_sum) > 0:
                color = colors[idx % len(colors)]
                clean_label = get_clean_label(sf)
                
                y_max = max_events_y * 1.15 if sf in event_sfs else shared_fat_top
                y_min = 0 if sf in event_sfs else shared_fat_bottom
                
                # Plot the line with markers enabled for hover but styled specifically
                fig.add_trace(go.Scatter(
                    x=subset[y_col],
                    y=subset[val_sum[0]],
                    mode='lines',
                    line=dict(color=color, width=2.5),
                    marker=dict(size=8, color="white", line=dict(color=color, width=2.5)), # Hollow white circle on hover
                    fill='tozeroy' if y_min == 0 else 'tonexty',
                    fillcolor=hex_to_rgba(color, 0.25),
                    name=clean_label,
                    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<extra></extra>',
                    hoverlabel=dict(bgcolor="white", font_size=13, bordercolor=color)
                ), row=row, col=col)
                
                fig.update_yaxes(range=[y_min, y_max], row=row, col=col)
                
                # Dynamic Y-axes logic: Only show labels and title on the first column
                if col == 1:
                    row_label = 'Number of Events' if row == 1 else 'Number of Deaths'
                    fig.update_yaxes(title_text=row_label, title_font=dict(size=12, color='#555555'), showticklabels=True, row=row, col=col)
                else:
                    fig.update_yaxes(showticklabels=False, row=row, col=col)
                
                fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', zeroline=False, tickformat=",d", tickfont=dict(color='#555555'), row=row, col=col)
                fig.update_xaxes(showline=True, linewidth=1.5, linecolor='lightgray', showgrid=False, tickmode='linear', dtick=2, tickfont=dict(color='#555555'), row=row, col=col)

                # Add a vertical dashed line for the 2024 turning point
                fig.add_shape(
                    type="line",
                    x0=2024, x1=2024,
                    y0=y_min, y1=y_max,
                    line=dict(color="#333333", width=1.2, dash="dash"),
                    opacity=0.7,
                    row=row, col=col
                )
                
                

        # Fix subplot titles: modify strictly the generated annotations
        for i, annot in enumerate(fig.layout.annotations):
            if i < len(subplot_titles):
                annot.font.color = colors[i % len(colors)]
                annot.font.size = 14
                annot.text = f"<b>{subplot_titles[i]}</b>"
                
        fig.update_layout(
            title=dict(
                text="<b>Trends of Domestic Violence and Fatalities in Iran (2016-2026)</b>",
                font=dict(size=20, color='black', family="sans-serif"),
                x=0.03,
                y=0.95
            ),
            showlegend=False,
            plot_bgcolor="white",
            paper_bgcolor="white",
            height=350 * n_rows,
            margin=dict(t=100, b=80, l=80, r=40),
            hovermode="closest" # Change from 'x unified' to closest to remove the vertical line
        )
        
        # Safely append footnote annotation instead of overwriting all annotations
        fig.add_annotation(
            text="Data covers reported ACLED events up to 2026 March.",
            showarrow=False,
            xref="paper", yref="paper",
            x=0.01, y=-0.15,
            font=dict(size=11, color="gray", style="italic")
        )
        
        # Explicit baseline trace to avoid fill bleeding when 'tonexty' is used without lower boundary
        for idx, sf in enumerate(unique_sources):
            if sf not in event_sfs:
                row = (idx // n_cols) + 1
                col = (idx % n_cols) + 1
                subset = yearly_data[yearly_data['Source_File'] == sf].sort_values(y_col)
                if not subset.empty:
                    fig.add_trace(go.Scatter(x=subset[y_col], y=[shared_fat_bottom]*len(subset), mode='lines', line=dict(color='rgba(0,0,0,0)', width=0), hoverinfo='skip', showlegend=False), row=row, col=col)
                    # Re-order the traces so the baseline is before the filled trace
                    fig.data = list(fig.data[:-2]) + [fig.data[-1], fig.data[-2]]
                
        trend_chart_html = fig.to_html(include_plotlyjs=False, full_html=False)
        fig.show()
    else:
        print("Required 'Year' or 'Value' columns missing for visualization.")
else:
    print(f"Data file not found at {out_path} for visualization.")

In [4]:
print(fig.layout.yaxis.title.text)
print(fig.layout.yaxis2.title.text)
print(fig.layout.yaxis4.title.text)

Number of Events
None
Number of Deaths


# 2. Iran Disobedience Level

In [5]:
import os
import pandas as pd

# Define paths (paths were previously defined, but just in case we redefine or use existing)
data_raw_path = os.path.join(project_path, "Data", "Data_Raw")
data_processed_path = os.path.join(project_path, "Data", "Data_Processed")

disobedience_file_name = "Middle-East_aggregated_data_up_to_week_of-2026-04-04.xlsx"
disobedience_file_path = os.path.join(data_raw_path, disobedience_file_name)
out_disobedience_path = os.path.join(data_processed_path, "Iran_disobedience.csv")

if os.path.exists(disobedience_file_path):
    print("Loading datasets...")
    # Load the big dataset
    df_disobedience = pd.read_excel(disobedience_file_path)
    
    # 1. Filter the dataset where the Country column is strictly equal to 'Iran'
    # Find the exact name of the country column (could be 'COUNTRY', 'Country', 'country')
    country_col = [c for c in df_disobedience.columns if c.lower() == 'country']
    
    if country_col:
        country_col_name = country_col[0]
        # Filter rows
        df_iran_disob = df_disobedience[df_disobedience[country_col_name] == 'Iran'].copy()
        
        # 2. Export to Data_Processed/Iran_disobedience.csv
        df_iran_disob.to_csv(out_disobedience_path, index=False)
        print(f"File successfully saved to: {out_disobedience_path}")
        print(f"Total rows extracted for Iran: {len(df_iran_disob)}")
        
        # 3. List unique items under the specified columns
        columns_to_list = ['ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'DISORDER_TYPE']
        
        for col in columns_to_list:
            # Match the exact column name case in the DataFrame
            actual_col = [c for c in df_iran_disob.columns if c.lower() == col.lower()]
            if actual_col:
                col_name = actual_col[0]
                unique_items = df_iran_disob[col_name].dropna().unique().tolist()
                print(f"\n--- Unique items in '{col_name}' ---")
                for item in sorted(unique_items):
                    print(f" - {item}")
            else:
                print(f"\n[Warning] Column '{col}' not found in the dataset.")
                
    else:
        print("Country column not found in the dataset.")
else:
    print(f"Disobedience data file not found at: {disobedience_file_path}")

Loading datasets...
File successfully saved to: /content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/Data/Data_Processed/Iran_disobedience.csv
Total rows extracted for Iran: 13478

--- Unique items in 'ADMIN1' ---
 - Alborz
 - Arabian Sea
 - Ardabil
 - Bushehr
 - Chaharmahal and Bakhtiari
 - East Azerbaijan
 - Fars
 - Gilan
 - Golestan
 - Gulf of Oman
 - Hamadan
 - Hormozgan
 - Ilam
 - Isfahan
 - Kerman
 - Kermanshah
 - Khuzestan
 - Kohgiluyeh and Boyer-Ahmad
 - Kurdistan
 - Lorestan
 - Markazi
 - Mazandaran
 - North Arabian Sea
 - North Khorasan
 - Persian Gulf
 - Qazvin
 - Qom
 - Razavi Khorasan
 - Semnan
 - Sistan and Baluchestan
 - South Khorasan
 - Strait of Hormuz
 - Tehran
 - West Azerbaijan
 - Yazd
 - Zanjan

--- Unique items in 'EVENT_TYPE' ---
 - Battles
 - Explosions/Remote violence
 - Protests
 - Riots
 - Strategic developments
 - Violence against civilians

--- Unique items in 'SUB_EVENT_TYPE' ---
 - Abductio

Data and Map successfully exported!
-> HTML Generated: /content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/Report/Iran_Disobedience_Report.html
-> Size maps to 'Events', Color maps to Normalized 'Index(0-1)'


# 3. Create Webpage

In [77]:
# Dashboard HTML Template
html_dashboard_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Iranian Civil Disobedience Dashboard</title>
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        :root {
            --bg-color: #f7f9fa;
            --text-primary: #14171a;
            --text-secondary: #657786;
            --card-bg: white;
            --border-color: #e1e8ed;
        }
        
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            margin: 0;
            padding: 0;
            background-color: var(--bg-color);
            color: var(--text-primary);
            line-height: 1.6;
        }

        .header {
            background-image: linear-gradient(rgba(0, 0, 0, 0.6), rgba(0, 0, 0, 0.8)), url('https://images.wsj.net/im-88861666?width=1280&height=853');
            background-size: cover;
            background-position: center;
            padding: 4rem 0;
            border-bottom: 1px solid var(--border-color);
            text-align: center;
            color: white;
        }

        .header h1 {
            margin: 0;
            font-size: 2.2rem;
            color: white;
            text-shadow: 1px 1px 3px rgba(0,0,0,0.8);
        }

        .header p {
            color: #e1e8ed;
            max-width: 800px;
            margin: 1rem auto 0;
            text-shadow: 1px 1px 3px rgba(0,0,0,0.8);
        }

        .container {
            max-width: 1200px;
            margin: 0 auto;
            padding: 2rem;
        }

        .narrative {
            background: var(--card-bg);
            padding: 2rem;
            border-radius: 8px;
            margin-bottom: 2rem;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }

        .narrative h2 {
            margin-top: 0;
            color: #2b3a4a;
        }
        
        .map-wrapper {
            background: var(--card-bg);
            padding: 1rem;
            border-radius: 8px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }

        #map {
            height: 600px;
            width: 100%;
            border-radius: 6px;
        }

        .footnotes {
            margin-top: 3rem;
            padding-top: 1rem;
            border-top: 1px solid var(--border-color);
            font-size: 0.9rem;
            color: var(--text-secondary);
        }

        /* Tooltip Styles */
        .tooltip-content {
            font-family: inherit;
            padding: 5px;
            min-width: 200px;
        }
        .tooltip-content h4 {
            margin: 0 0 10px 0;
            font-size: 16px;
            color: #333;
            border-bottom: 1px solid #ddd;
            padding-bottom: 5px;
        }
        .region-label {
            font-size: 10px;
            text-transform: uppercase;
            color: #777;
            margin: 0 0 2px 0;
        }
        .score-label {
            font-weight: bold;
            font-size: 13px;
            margin-bottom: 15px;
        }
        .pie-chart {
            width: 100%;
            height: 120px;
            border-radius: 50%;
            margin-bottom: 15px;
            box-shadow: 0 0 5px rgba(0,0,0,0.2) inset;
        }
        .legend {
            font-size: 11px;
            display: grid;
            grid-template-columns: 1fr 1px 1fr;
            gap: 5px;
        }
        .legend-item {
            display: flex;
            align-items: center;
        }
        .color-box {
            width: 10px;
            height: 10px;
            margin-right: 6px;
            border-radius: 2px;
        }
    </style>
</head>
<body>

<div class="header">
    <h1>Betting on the Street: Can Iranian People Dissent Flip the House Odds?</h1>
    <p>A geospatial analysis of political violence, demonstrations, and shifting modes of unrest from 2016 to 2026. This dashboard examines regional hotspots and the transition in event types.</p>
</div>

<div class="container">
    <div class="narrative">
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">Introduction</h2>
        <p>The ongoing US-Iranian conflict has disturbed the global economy, driving fuel prices to rise subst antially. Despite various diplomatic efforts, a lasting resolution remains elusive. Following the assassination of Ali Khamenei on 28 February, the expectation of Iranian regime change has reached a fever pitch. 
        It appears that the United States and Israel have deployed strategic air strikes, political pressure, and the incitement of internal uprisings to destabilize or overthrow the IRGC regime. Unfortunately, while the U.S. has destroyed Iran’s conventional air and naval capabilities, the IRGC regime maintains its grip through the control of the Strait of Hormuz and brutal suppression of domestic unrest. </p>
        <p>Liberal hawks such as Kenneth M. Pollack advocate for moves such as direct air support to an Iranian popular revolt to increase the odds of winning the conflict. According to him, Trump's military campaign gambles on 'whether a massive air campaign can trigger a popular rebellion that takes down the regime in Tehran.'<sup id="ref1"><a href="#fn1">[1]</a></sup></p>
        <p>It thus becomes vital to study the Iranian people's disobedience tendency and their resistance to the government. The author examines the patterns in civil disobedience and resistance movements across Iran in the last decade. This report also aims to use geospatial data science to identify strategic clues: which regions are currently experiencing the highest friction, which cities serve as the most viable starting points for unrest, and where the regime's control is most contested.</p>
        
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">An Underlying Tide of Resistance</h2>
        <p>The data reveals a significant structural shift in the nature of domestic unrest and state-society relations in Iran starting in 2024. While outward signs of mass mobilization have declined, indicators of organized resistance and state-level violence have reached historic highs. The interactive trend charts below trace this explosion of domestic violence and fatalities, marking 2024 as a critical turning point where both peaceful protests and violent demonstrations sharply increased.<sup id="ref2"><a href="#fn2">[2]</a></sup></p>
        <p>The sharp decline in Demonstration Events after 2024 does not suggest a restoration of social stability or a decrease in public grievances. Instead, it reflects a "chilling effect" resulting from extreme state suppression. The high cost of public assembly—characterized by mass arrests and lethal force—has forced the opposition to retreat from the streets.<p>
        <p>Critically, the upward trend in Strategic Developments and Events Targeting Civilians provides strong circumstantial evidence that opposition forces are undergoing a qualitative upgrade. In ACLED terms, "Strategic Developments" often involve recruitment, weapon movements, and the establishment of underground networks. The rise in this metric suggests that the opposition is moving away from uncoordinated street rallies toward organized, clandestine resistance. The increase in events targeting civilians—whether conducted by the state to instill fear or by radicalized opposition cells—indicates a breakdown of the social contract and a shift toward asymmetric conflict.</p>
        
        <!-- Inject Plotly interactive chart here -->
        <div class="trend-chart-container" style="margin: 2rem 0; width: 100%; border: 1px solid #e1e8ed; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); background: white;">
            __TREND_CHART_HTML__
        </div>
    <div>
  
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">Disobedience Levels in Regions</h2>
        <p>
        To move beyond simple event counts, this analysis develops a <strong>Disobedience Severity Index (DSI)</strong> to quantify the intensity of regional unrest. 
        By weighting events based on the degree of state-citizen friction, this index helps identify "breakthrough points" where the social contract has effectively collapsed. 
        The weights are assigned as follows:
        </p>
        <p><strong>1. Categorizing Resistance Sub-types</strong></p>
        
    

    <table class="dsi-table">
        <thead>
            <tr>
                <th>ACLED Event Type</th>
                <th>Description</th>
                <th>Weight (w)</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td><strong>Peaceful Protest</strong></td>
                <td>Non-violent assembly without state interference; represents baseline dissent.</td>
                <td>1</td>
            </tr>
            <tr>
                <td><strong>Protest with Intervention</strong></td>
                <td>Non-violent assembly met with state attempts to disperse or arrest participants.</td>
                <td>2</td>
            </tr>
            <tr>
                <td><strong>Excessive Force against Protesters</strong></td>
                <td>State actors use lethal or disproportionate force against non-violent protesters.</td>
                <td>3</td>
            </tr>
            <tr>
                <td><strong>Violent Demonstration</strong></td>
                <td>Kinetic events involving rioting or armed clashes; indicates a transition to active rebellion.</td>
                <td>4</td>
            </tr>
        </tbody>
    </table>
</div>

<style>
    /* Styling for the Methodology Table */
    .dsi-table {
        width: 100%;
        border-collapse: collapse;
        margin: 20px 0;
        background-color: white;
        border-radius: 8px;
        overflow: hidden;
        box-shadow: 0 2px 5px rgba(0,0,0,0.05);
    }
    
    .dsi-table th {
        background-color: #f1f3f5;
        color: #2c3e50;
        text-align: left;
        padding: 12px 15px;
        border-bottom: 2px solid #dee2e6;
        font-size: 0.9rem;
        text-transform: uppercase;
        letter-spacing: 0.05em;
    }

    .dsi-table td {
        padding: 12px 15px;
        border-bottom: 1px solid #edf2f7;
        font-size: 0.95rem;
        color: #4a5568;
        vertical-align: middle;
    }

    .dsi-table tr:last-child td {
        border-bottom: none;
    }

    /* Highlight the Weight Column */
    .dsi-table td:last-child {
        font-weight: bold;
        color: #e74c3c;
        text-align: center;
        background-color: #fffaf0;
    }

    .dsi-table tr:hover {
        background-color: #f8fafc;
    }
</style>



    <h3 style="font-size: 1.1rem; color: #444;">2. Index Calculation</h3>
    <p>For each record, the <em>Disobedience Score</em> is calculated using the following formula to penalize high-fatality and high-weight events while normalizing for population density:</p>
    
    <div style="background: #f4f4f4; padding: 20px; border-radius: 5px; text-align: center; font-family: 'Courier New', Courier, monospace; font-size: 1.2rem; margin: 20px 0;">
        Score = (w<sup>2</sup> × Events × Fatalities) / Population Exposure
    </div>

    <p>The resulting scores are then aggregated by province (Admin1) and normalized to a <strong>0.0 – 1.0 scale</strong> (where 1.0 represents the highest intensity recorded in the dataset), creating a comparative <strong>Disobedience Index</strong> across Iran.</p>

    <h3 style="font-size: 1.1rem; color: #444; margin-top: 2rem;">3. Visual Interpretation & Geospatial Distribution</h3>
    <p style="color: #555;">
        The interactive map below translates the DSI methodology into a spatial intelligence tool. By analyzing the relationship between volume and intensity, we can pinpoint the "breakthrough points" mentioned previously:
    </p>
    
    <ul style="color: #555; margin-bottom: 2rem;">
        <li><strong>Bubble Size:</strong> Represents the <em>Total Volume</em> of events, identifying mobilization hubs.</li>
        <li><strong>Bubble Color:</strong> Represents the <em>Index Severity</em> (DSI). Deeper shades of red indicate regions where the social contract has reached a terminal breaking point due to excessive force or lethal clashes.</li>
        <li><strong>Interactive Elements:</strong> Hover over any province to trigger a <strong>Sub-event Pie Chart</strong>, which reveals the local ratio of peaceful dissent vs. violent rebellion.</li>
    </ul>

    <div class="map-wrapper" style="margin: 2rem 0; padding: 10px; background: white; border: 1px solid #e1e8ed; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
        <div id="map" style="height: 600px; width: 100%; border-radius: 8px;"></div>
        <p style="margin-top: 1rem; font-size: 0.85rem; color: var(--text-secondary); text-align: center; font-style: italic;">
            Figure 1: Interactive Geospatial Analysis of the Disobedience Severity Index (DSI) across Iran (2016-2026).
        </p>
    </div>


    <div class="conclusion-section" style="margin-top: 3rem;">
        <h2 style="border-left: 4px solid #cb181d; padding-left: 15px; color: #2a3f5f;">Looking Ahead: The Strategy of Precision Support</h2>
        <p>
            The findings from this dashboard suggest that a uniform intervention strategy would likely fail. Instead, the data points toward a <strong>decentralized approach</strong>. External actors should look beyond the total event counts in Tehran and prioritize support for regions like <strong>Isfahan</strong> and <strong>Sistan and Baluchestan</strong>, where the "Disobedience Index" and "Violent Proportion" clues suggest the existing domestic friction is ripe for escalation.
        </p>
        <p>
            Future analysis should integrate IRGC deployment densities with this DSI map to identify "security vacuums"—areas where high citizen disobedience meets low state military presence—as the ultimate breakthrough targets.
        </p>
    </div>


    

    <div class="footnotes">
        <p id="fn1"><strong>[1]</strong> <em>How to Raise the Odds of Regime Change in Iran: America Can Make It Easier for Iranians to Revolt</em>, Kenneth M. Pollack, March 18, 2026, <a href="https://www.foreignaffairs.com/iran/how-raise-odds-regime-change-iran" target="_blank">Foreign Affairs</a>. <a href="#ref1">↩</a></p>
        <p id="fn2"><strong>[2]</strong> The data relies on millions of ACLED (Armed Conflict Location & Event Data Project) points up to March 2026. Sub-category filters for political violence and demonstration events were strictly aggregated to map regional severity. <a href="#ref2">↩</a></p>
        <p style="margin-top: 1rem;"><strong>Map Visualization Note:</strong> The region bubbles visualize peaceful protests (Blue), state interventions (Orange), excessive force (Purple), and violent clashes (Red). The exact percentage split is shown in the tooltip pie chart when hovering over regions.</p>
    </div>
</div>

<script>
    var mapData = __DASHBOARD_JSON_DATA__;
    
    // Initialize map focused on Iran
    var map = L.map('map').setView([32.4279, 53.6880], 5);
    
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', {
        attribution: '&copy; CARTO', subdomains: 'abcd', maxZoom: 20
    }).addTo(map);
    
    mapData.sort(function(a, b) { return b.TOTAL_EVENTS - a.TOTAL_EVENTS; });
    
    var maxEvents = Math.max(...mapData.map(d => d.TOTAL_EVENTS));
    function getRadius(val, max) { return 5 + Math.sqrt(val / max) * 30; }

    function getRedColor(ratio) {
        var pal = ['#fff5f0', '#fee0d2', '#fcbba1', '#fc9272', '#fb6a4a', '#ef3b2c', '#cb181d', '#99000d'];
        var idx = Math.floor(ratio * (pal.length - 1));
        if(idx < 0) idx = 0; if(idx >= pal.length) idx = pal.length - 1;
        return pal[idx];
    }
    
    mapData.forEach(function(region) {
        var size = getRadius(region.TOTAL_EVENTS, maxEvents);
        var color = getRedColor(region.DISOBEDIENCE_INDEX);
        
        var circle = L.circleMarker([region.LAT, region.LONG], {
            radius: size, fillColor: color, color: "#cb181d",   
            weight: 1.5, opacity: 0.8, fillOpacity: 0.75
        }).addTo(map);

        circle.on('mouseover', function() {
            this.setStyle({ fillOpacity: 1, weight: 2.5, color: "#333" });
            this.bringToFront();
        });
        circle.on('mouseout', function() {
            this.setStyle({ fillOpacity: 0.75, weight: 1.5, color: "#cb181d" });
        });

        var p = region.peaceful || 0, i = region.intervention || 0, f = region.force || 0, v = region.violent || 0;
        var c_p = '#3498db', c_i = '#f39c12', c_f = '#8e44ad', c_v = '#e74c3c';

        var bg = 'conic-gradient(' + 
            c_p + ' 0% ' + p + '%, ' +                             
            c_i + ' ' + p + '% ' + (p+i) + '%, ' +                 
            c_f + ' ' + (p+i) + '% ' + (p+i+f) + '%, ' +           
            c_v + ' ' + (p+i+f) + '% 100%)';                       

        var tooltipHtml = `<div class="tooltip-content">
            <p class="region-label">Region / Province</p>
            <h4>${region.ADMIN1}</h4>
            <p class="score-label">Events: ${region.TOTAL_EVENTS}<br>Index: ${region.DISOBEDIENCE_INDEX.toFixed(2)}</p>
            <div class="pie-chart" style="background: ${bg};"></div>
            <div class="legend">
                <div class="legend-item"><div class="color-box" style="background:${c_p};"></div>Peaceful</div>
                <div class="legend-item"><div class="color-box" style="background:${c_i};"></div>Intervention</div>
                <div class="legend-item"><div class="color-box" style="background:${c_f};"></div>Force</div>
                <div class="legend-item"><div class="color-box" style="background:${c_v};"></div>Violent</div>
            </div>
        </div>`;

        circle.bindTooltip(tooltipHtml, {
            sticky: true,
            className: 'custom-tooltip',
            opacity: 0.95
        });
    });
</script>
</body>
</html>
"""

# Ensure react_data_json from the previous cell is used
dashboard_html_content = html_dashboard_template.replace("__DASHBOARD_JSON_DATA__", react_data_json).replace("__TREND_CHART_HTML__", trend_chart_html)
dashboard_out_path = os.path.join(project_path, "Report", "Dashboard.html")

with open(dashboard_out_path, "w", encoding="utf-8") as f:
    f.write(dashboard_html_content)

print(f"Dashboard successfully created and exported to: {dashboard_out_path}")

Dashboard successfully created and exported to: /content/drive/MyDrive/Study/HKU/POLI3148 Data Science in Politics and Public Administration/Assignments/Assignment 1/Report/Dashboard.html
